In [1]:
# Python — Google Colab
# ============================================================
# CELL 1 — Instalasi Library
# Jalankan cell ini SEKALI di awal sesi baru
# Waktu instalasi: 2–5 menit
# ============================================================

# Install library utama
!pip install earthengine-api --quiet   # Google Earth Engine Python API
!pip install geemap --quiet            # Wrapper interaktif untuk GEE

# Verifikasi instalasi
import ee
import geemap
print(f"earthengine-api versi: {ee.__version__}")
print(f"geemap versi: {geemap.__version__}")
print("✅ Instalasi berhasil!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.8 MB/s eta 0:00:00
earthengine-api versi: 1.7.39
geemap versi: 0.38.3
✅ Instalasi berhasil!


In [2]:
# Python — Google Colab
# ============================================================
# CELL 2 — Autentikasi dan Inisialisasi GEE
# ============================================================

import ee
import geemap

# Langkah 1: Autentikasi
# Akan muncul link — klik link tersebut, login dengan akun GEE Anda,
# salin kode verifikasi, paste ke kotak yang muncul di bawah cell ini.
ee.Authenticate()

# Langkah 2: Inisialisasi
# Ganti "your-project-id" dengan ID project GEE Anda.
# Jika tidak punya project ID, gunakan ee.Initialize() saja tanpa argumen.
ee.Initialize(project="random-forest-cangg")

# Verifikasi koneksi
print("Status koneksi GEE:", ee.String("Berhasil terhubung!").getInfo())
print("✅ GEE siap digunakan!")


Status koneksi GEE: Berhasil terhubung!
✅ GEE siap digunakan!


In [3]:
# Python — Google Colab
# ============================================================
# CELL 3 — Import Semua Library yang Dibutuhkan
# ============================================================

import ee          # Google Earth Engine API
import geemap      # Peta interaktif berbasis GEE
import json        # Untuk parsing JSON (nanti digunakan untuk AOI)

print("📦 Semua library berhasil diimport!")


📦 Semua library berhasil diimport!


In [4]:
# Python — Google Colab
# ============================================================
# CELL 4 — Membuat Peta Interaktif dan Menampilkan Citra Landsat
# ============================================================

# Buat objek peta interaktif geemap
# geemap.Map() menggabungkan GEE dengan visualisasi Leaflet.js
Map = geemap.Map()

# Tentukan area kajian (ganti koordinat sesuai AOI proyek Anda!)
# Format: ee.Geometry.Point([longitude, latitude])
aoi_point = ee.Geometry.Point([119.4328, -5.1477])  # Makassar sebagai contoh

# Load ImageCollection Landsat 9
landsat_collection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2") \
    .filterDate("2023-01-01", "2023-12-31") \
    .filterBounds(aoi_point) \
    .filterMetadata("CLOUD_COVER", "less_than", 15)

# Cek berapa banyak citra tersedia
jumlah_citra = landsat_collection.size().getInfo()
print(f"📊 Jumlah citra tersedia: {jumlah_citra} scene")

# Ambil citra dengan cloud cover terendah (kualitas terbaik)
citra_terbaik = landsat_collection \
    .sort("CLOUD_COVER") \
    .first()

# Tampilkan tanggal akuisisi
tanggal = citra_terbaik.date().format("YYYY-MM-dd").getInfo()
print(f"📅 Tanggal akuisisi citra terbaik: {tanggal}")

# Parameter visualisasi True Color
vis_true = {
    "bands": ["SR_B4", "SR_B3", "SR_B2"],
    "min": 7000,
    "max": 13000,
    "gamma": 1.4
}

# Parameter visualisasi False Color (NIR-Red-Green)
vis_false = {
    "bands": ["SR_B5", "SR_B4", "SR_B3"],
    "min": 7000,
    "max": 20000
}

# Tambahkan layer ke peta
Map.centerObject(aoi_point, 10)
Map.addLayer(citra_terbaik, vis_true,  "Landsat9 True Color")
Map.addLayer(citra_terbaik, vis_false, "Landsat9 False Color", shown=False)
Map.addLayer(aoi_point, {"color": "FF0000"}, "Titik AOI", shown=True)

# Tampilkan peta
Map


📊 Jumlah citra tersedia: 10 scene
📅 Tanggal akuisisi citra terbaik: 2023-09-06


Map(center=[-5.1477, 119.43280000000003], controls=(WidgetControl(options=['position', 'transparent_bg'], posi…

In [5]:
# Python — Google Colab
# ============================================================
# CELL 5 — Menampilkan Metadata Citra
# ============================================================

# Ambil semua properti (metadata) citra
info_citra = citra_terbaik.getInfo()

# Tampilkan informasi penting
properties = citra_terbaik.toDictionary()

print("=" * 50)
print("INFORMASI CITRA LANDSAT 9")
print("=" * 50)

cloud_cover = citra_terbaik.get("CLOUD_COVER").getInfo()
spacecraft  = citra_terbaik.get("SPACECRAFT_ID").getInfo()
path        = citra_terbaik.get("WRS_PATH").getInfo()
row         = citra_terbaik.get("WRS_ROW").getInfo()

print(f"Satelit      : {spacecraft}")
print(f"Tanggal      : {tanggal}")
print(f"Path/Row     : {path}/{row}")
print(f"Cloud Cover  : {cloud_cover:.1f}%")

# Tampilkan nama-nama band
band_names = citra_terbaik.bandNames().getInfo()
print(f"Band tersedia: {band_names}")
print("=" * 50)


INFORMASI CITRA LANDSAT 9
Satelit      : LANDSAT_9
Tanggal      : 2023-09-06
Path/Row     : 114/64
Cloud Cover  : 2.1%
Band tersedia: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [6]:
# Python — Google Colab
# ============================================================
# CELL 6 — Sentinel-2 Sebagai Alternatif (Resolusi 10m)
# ============================================================

# Sentinel-2 Surface Reflectance — resolusi lebih tinggi (10m vs 30m Landsat)
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterDate("2023-01-01", "2023-12-31") \
    .filterBounds(aoi_point) \
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)) \
    .sort("CLOUDY_PIXEL_PERCENTAGE") \
    .first()

# Cek tanggal
tgl_s2 = sentinel2.date().format("YYYY-MM-dd").getInfo()
cloud_s2 = sentinel2.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
print(f"Sentinel-2 terbaik: {tgl_s2} (cloud: {cloud_s2:.1f}%)")

# Visualisasi Sentinel-2 True Color
vis_s2_true = {
    "bands": ["B4", "B3", "B2"],  # Band S2 berbeda dari Landsat!
    "min": 0,
    "max": 3000
}

# Tambahkan ke peta yang sama
Map.addLayer(sentinel2, vis_s2_true, "Sentinel-2 True Color", shown=False)
Map  # Tampilkan ulang peta dengan layer baru


Sentinel-2 terbaik: 2023-09-21 (cloud: 0.0%)


Map(bottom=34092.0, center=[-5.615985819155327, 119.13024902343751], controls=(WidgetControl(options=['positio…

In [7]:
# Python — Google Colab
# ============================================================
# CELL 7 — Mendefinisikan dan Menyimpan AOI Proyek
# ============================================================

import ee
import geemap
import json

# ── Opsi A: Definisikan AOI dengan koordinat manual ──────────
# Ganti koordinat ini dengan wilayah kajian Anda!
aoi_coords = [
    [119.20, -5.50],  # SW
    [119.80, -5.50],  # SE
    [119.80, -4.90],  # NE
    [119.20, -4.90],  # NW
    [119.20, -5.50]   # kembali ke SW (tutup polygon)
]

# Buat objek Geometry GEE
aoi = ee.Geometry.Polygon([aoi_coords])

# ── Opsi B: AOI dari bounding box ────────────────────────────
# aoi = ee.Geometry.Rectangle([lon_barat, lat_selatan, lon_timur, lat_utara])
# aoi = ee.Geometry.Rectangle([119.2, -5.5, 119.9, -4.8])

# ── Informasi AOI ─────────────────────────────────────────────
luas_m2  = aoi.area(maxError=1).getInfo()
luas_km2 = luas_m2 / 1_000_000
print(f"📍 AOI berhasil didefinisikan")
print(f"📐 Luas AOI: {luas_km2:.2f} km²")

# ── Visualisasi AOI di peta ───────────────────────────────────
Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(
    aoi,
    {"color": "FF0000", "fillColor": "FF000020"},
    "AOI Proyek"
)
Map


📍 AOI berhasil didefinisikan
📐 Luas AOI: 4432.86 km²


Map(center=[-5.200023235010828, 119.49999999999939], controls=(WidgetControl(options=['position', 'transparent…

In [8]:
# ==========================================
# PYTHON - GOOGLE COLAB
# Simpan AOI sebagai GeoJSON ke Google Drive
# ==========================================

from google.colab import drive
import json
import os

# Mount Google Drive
drive.mount("/content/drive")

# Folder utama di Google Drive
folder_proyek = "/content/drive/MyDrive/ProyekGEE_PemSpasial_FAUZAN"

# Folder untuk data AOI
folder_aoi = os.path.join(folder_proyek, "data_aoi")

# Buat folder
os.makedirs(folder_aoi, exist_ok=True)

# ==========================================
# Konversi AOI menjadi GeoJSON
# ==========================================

aoi_geojson = aoi.getInfo()

geojson_output = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {
            "nama": "AOI_Proyek_PemSpasial",
            "mahasiswa": "MUH. FAUZAN",
            "nim": "V126241004",
            "deskripsi": "Area kajian proyek Pemrograman Spasial 2026"
        },
        "geometry": aoi_geojson
    }]
}

# ==========================================
# Simpan ke Google Drive
# ==========================================

path_output = os.path.join(
    folder_aoi,
    "AOI_Proyek_Pemrograman_Spasial.geojson"
)

with open(path_output, "w") as f:
    json.dump(geojson_output, f, indent=2)

print("✅ AOI berhasil disimpan!")
print("📁 Lokasi:")
print(path_output)
print("📦 Ukuran file:", os.path.getsize(path_output), "bytes")

Mounted at /content/drive
✅ AOI berhasil disimpan!
📁 Lokasi:
/content/drive/MyDrive/ProyekGEE_PemSpasial_FAUZAN/data_aoi/AOI_Proyek_Pemrograman_Spasial.geojson
📦 Ukuran file: 753 bytes


In [9]:
# Python — Google Colab
# ============================================================
# CELL 9 — Upload AOI ke GEE Assets (Opsional tapi Disarankan)
# ============================================================

# Cara 1: Upload menggunakan geemap (lebih mudah)
# geemap.shp_to_ee() untuk shapefile
# atau export via ee.batch.Export untuk dari Colab

# Export AOI sebagai FeatureCollection ke GEE Assets
aoi_feature = ee.Feature(aoi, {
    "nama": "AOI_Proyek",
    "mahasiswa": "MUH. FAUZAN",
    "nim": "V126241004"
})

aoi_fc = ee.FeatureCollection([aoi_feature])

# Ganti "your_username" dengan username GEE Anda
task = ee.batch.Export.table.toAsset(
    collection=aoi_fc,
    description="Upload_AOI_Proyek",
    assetId="projects/earthengine-legacy/assets/users/MUHFAUZAN/forest-canggg"
)

task.start()
print("🚀 Task upload AOI dimulai!")
print("Cek status di GEE Code Editor → Tasks")

# Monitor status task
import time
status = task.status()
print(f"Status awal: {status['state']}")


🚀 Task upload AOI dimulai!
Cek status di GEE Code Editor → Tasks
Status awal: READY


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Folder utama proyek
project = "/content/drive/MyDrive/ProyekGEE_PemSpasial_FAUZAN"

# Daftar folder yang diperlukan
folders = [
    "data_aoi",
    "data_raw",
    "data_processed",
    "notebooks",
    "gee_scripts",
    "outputs/maps",
    "outputs/stats",
    "outputs/charts",
    "report"
]

# Membuat semua folder
for folder in folders:
    path = os.path.join(project, folder)
    os.makedirs(path, exist_ok=True)

print("✅ Struktur folder berhasil dibuat!")
print(project)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Struktur folder berhasil dibuat!
/content/drive/MyDrive/ProyekGEE_PemSpasial_FAUZAN


In [ ]:
# Python — Google Colab
# ============================================================
# CELL 10 — Koneksi Colab ke GitHub (Commit Pertama)
# ============================================================

# Konfigurasi identitas git
!git config --global user.email "muhammadfauzaan2606-debug@gmail.com"
!git config --global user.name "MUH. FAUZAN"

# Clone repository dari GitHub ke Colab
# Ganti URL dengan URL repository GitHub Anda
!git clone https://github.com/muhammadfauzaan2606-debug/PemSpasial_GEE_V126241004.git

# Masuk ke folder repository
%cd PemSpasial_GEE_V126241004

# Buat struktur folder
!mkdir -p data_aoi data_raw data_processed notebooks gee_scripts outputs/maps outputs/stats outputs/charts report

# Salin notebook dari lokasi Colab ke folder repository
!cp /content/drive/MyDrive/ProyekGEE_PemSpasial/data_aoi/AOI_proyek.geojson data_aoi/

# Tambahkan file ke git dan commit
!git add .
!git commit -m "Pertemuan 1: Setup project structure, AOI defined"

# Push ke GitHub (perlu personal access token)
# Ganti URL dengan URL repo Anda dan masukkan token jika diminta
!git push origin main

print("✅ Commit pertama berhasil!")


## Tugas A — Temukan Filter Optimal untuk AOI

| Periode | Sensor | Cloud < 20% | Cloud < 30% | Filter Optimal Pilihan Anda |
|---|---|---:|---:|---|
| T1: 2019–2020 | Landsat 8 | 15 scene | 22 scene | <30%, karena jumlah scene lebih banyak |
| T2: 2022–2023 | Landsat 9 | 12 scene | 18 scene | <30%, karena jumlah scene lebih banyak |

### Kesimpulan Filter Optimal

Berdasarkan hasil eksplorasi data Landsat di Google Earth Engine, filter cloud cover **<30%** dipilih sebagai filter optimal. Filter tersebut memberikan keseimbangan antara jumlah scene yang tersedia dan kualitas citra. Meskipun filter **<20%** menghasilkan citra dengan tutupan awan yang lebih rendah, jumlah scene yang tersedia lebih sedikit. Dengan menggunakan filter **<30%**, jumlah data yang dapat digunakan untuk membuat composite menjadi lebih banyak sehingga dapat meningkatkan ketersediaan data untuk analisis.

## Tugas C — Laporan Eksplorasi Data

### 1. Jumlah Scene T1 dan T2

Pada periode T1 (2017–2018) tersedia sebanyak **22 scene Landsat 8**, sedangkan pada periode T2 (2022–2023) tersedia sebanyak **27 scene Landsat 9**. Jumlah scene tersebut cukup untuk membuat composite karena tersedia banyak citra pada masing-masing periode. Composite median dapat digunakan untuk menggabungkan citra dan mengurangi pengaruh awan serta gangguan lainnya pada citra individual.

### 2. Rata-rata Cloud Cover T1 dan T2

Rata-rata cloud cover pada T1 sebesar **22,48%**, sedangkan pada T2 sebesar **22,06%**. Kondisi ini tergolong cukup baik untuk analisis karena kedua periode telah difilter menggunakan batas cloud cover kurang dari 30%. T2 memiliki rata-rata cloud cover sedikit lebih rendah dibandingkan T1, sehingga kondisi tutupan awannya sedikit lebih baik.

### 3. Rata-rata NDVI di AOI

Nilai rata-rata NDVI pada T1 sebesar **0,7778**, sedangkan pada T2 sebesar **0,7867**. Nilai NDVI yang berada di sekitar 0,78 menunjukkan bahwa wilayah Kabupaten Luwu Timur secara umum memiliki tutupan vegetasi yang tinggi atau relatif lebat. Nilai NDVI T2 sedikit lebih tinggi dibandingkan T1, yang menunjukkan adanya peningkatan kecil pada nilai rata-rata tutupan vegetasi pada periode T2.

### 4. Dataset Publik GEE yang Relevan

Dataset publik GEE yang paling relevan untuk mendukung proyek adalah **Dynamic World (`GOOGLE/DYNAMICWORLD/V1`)**. Dataset ini menyediakan informasi tutupan lahan dengan beberapa kelas seperti air, vegetasi, permukiman, dan lahan terbuka. Oleh karena itu, Dynamic World dapat digunakan sebagai data pendukung untuk membantu identifikasi dan pemilihan training sample. Sementara itu, Landsat 8 dan Landsat 9 digunakan sebagai data utama untuk analisis citra, pembuatan composite, dan perhitungan NDVI.

# 🗓️ Jurnal Harian — Pertemuan 2

**Tanggal:** 7 September 2026 | **Nama:** MUH. FAUZAN | **NIM:** V126241004 | **AOI Proyek:** Kabupaten Luwu Timur

### 🛰️ Status Proyek Hari Ini

| Komponen | Status | Catatan |
|---|---|---|
| AOI terdefinisi | ✅ | Kabupaten Luwu Timur |
| Composite T1 dibuat | ✅ | Landsat 8, 2017–2018, 22 scene |
| Composite T2 dibuat | ✅ | Landsat 9, 2022–2023, 27 scene |
| Titik training dibuat | ✅ | 40 titik, 4 kelas |
| Ekspor ke Drive | ✅ | Training sample diekspor dalam format CSV |

### 📊 Data yang Ditemukan

- **Jumlah scene T1:** 22 scene (Landsat 8, 2017–2018)
- **Jumlah scene T2:** 27 scene (Landsat 9, 2022–2023)
- **Cloud cover rata-rata T1:** 22,48%
- **Cloud cover rata-rata T2:** 22,06%
- **Rata-rata NDVI T1:** 0,7778
- **Rata-rata NDVI T2:** 0,7867

### ✅ Yang berhasil hari ini

Script untuk eksplorasi data dan pembuatan training sample berhasil dijalankan di Google Earth Engine. Hasil yang diperoleh berupa composite Landsat 8 untuk T1 dan Landsat 9 untuk T2, nilai rata-rata cloud cover dan NDVI, serta 40 training sample yang terdiri dari 4 kelas, yaitu Air, Vegetasi, Permukiman, dan Lahan Terbuka.

### ❌ Error yang ditemui

Tidak terdapat error yang menghambat proses. Script berhasil dijalankan dan menghasilkan data sesuai dengan tujuan eksplorasi.

### 💡 Keputusan teknis yang diambil

- **Sensor yang dipilih (Landsat/Sentinel):** Landsat 8 dan Landsat 9 karena memiliki data multispektral yang sesuai untuk analisis tutupan lahan dan perhitungan NDVI.
- **Threshold cloud cover:** < 30% karena digunakan untuk menyaring citra dengan tutupan awan yang terlalu tinggi.
- **Periode T1:** 2017–2018 karena digunakan sebagai periode awal untuk perbandingan.
- **Periode T2:** 2022–2023 karena digunakan sebagai periode pembanding.
- **Kelas tutupan lahan yang dipilih:** Air, Vegetasi, Permukiman, dan Lahan Terbuka karena merupakan kelas yang digunakan dalam training sample.

### 🔭 Temuan awal tentang wilayah kajian

Berdasarkan nilai rata-rata NDVI sebesar **0,7778 pada T1** dan **0,7867 pada T2**, Kabupaten Luwu Timur secara umum memiliki tutupan vegetasi yang tinggi atau relatif lebat. Nilai NDVI pada T2 sedikit lebih tinggi dibandingkan T1, sehingga terdapat peningkatan kecil pada nilai rata-rata vegetasi antara kedua periode.

### ⭐ Hal paling menarik dari pertemuan ini

Hal yang paling menarik adalah dapat melihat bagaimana citra Landsat dari beberapa scene dapat digabungkan menjadi composite dan digunakan untuk menghitung NDVI. Selain itu, Dynamic World dapat dimanfaatkan sebagai data pendukung untuk membantu menentukan training sample berdasarkan kelas tutupan lahan.

Berdasarkan hasil perhitungan rata-rata cloud cover bulanan pada wilayah Kabupaten Luwu Timur, bulan Februari memiliki citra paling bersih dari awan, dengan rata-rata cloud cover sebesar 17,40%. Nilai ini merupakan yang paling rendah dibandingkan bulan lainnya. Kondisi tersebut menunjukkan bahwa Februari memiliki peluang yang lebih baik untuk memperoleh citra Landsat dengan tutupan awan yang relatif rendah. Kondisi ini cukup sesuai dengan pola curah hujan karena periode yang lebih kering umumnya cenderung memiliki tutupan awan yang lebih rendah. Namun, pola curah hujan di Kabupaten Luwu Timur dapat bervariasi sehingga hubungan antara curah hujan dan cloud cover tidak selalu sama setiap tahun.

## Tugas D — Eksplorasi Konsep dan Kualitas Citra

### 1. Perbedaan ee.Image dan ee.ImageCollection
Bayangkan sebuah album foto. `ee.Image` seperti satu lembar foto yang menggambarkan kondisi suatu wilayah pada satu waktu. Sedangkan `ee.ImageCollection` seperti sebuah album yang berisi banyak foto dari wilayah yang sama tetapi diambil pada waktu yang berbeda.

### 2. Mengapa NDVI bisa negatif?
NDVI dapat bernilai negatif karena beberapa permukaan tidak memiliki karakteristik pantulan seperti vegetasi. Di Kabupaten Luwu Timur, nilai NDVI negatif kemungkinan besar ditemukan pada permukaan air seperti danau, sungai, atau wilayah tergenang. Vegetasi sehat umumnya memiliki nilai NDVI positif dan tinggi.

### 3. Bulan dengan citra paling bersih
Berdasarkan hasil yang telah diperoleh, rata-rata cloud cover T1 adalah 22,48% dan T2 adalah 22,06%. Namun, bulan dengan citra paling bersih belum dapat ditentukan dari hasil tersebut karena cloud cover yang dihitung merupakan rata-rata seluruh scene. Bulan dengan nilai rata-rata cloud cover paling rendah dapat ditentukan dengan menghitung cloud cover setiap bulan. Secara umum, periode dengan curah hujan lebih rendah cenderung memiliki kondisi citra yang lebih bersih dari awan.

### 4. Penggunaan .first()
Jika `.first()` digunakan tanpa `.sort("CLOUD_COVER")`, citra yang diambil adalah citra pertama dalam koleksi dan tidak menjamin memiliki cloud cover paling rendah. Oleh karena itu, citra tersebut dapat lebih berawan. Untuk memilih citra yang lebih bersih dapat digunakan `.sort("CLOUD_COVER").first()`.

### 5. Saran untuk AOI pegunungan Papua
Jika wilayah pegunungan Papua selalu tertutup awan, sebaiknya menggunakan periode pencarian yang lebih panjang, melakukan cloud masking, dan membuat composite dari banyak citra menggunakan metode seperti median. Jika citra optik tetap sulit diperoleh karena awan, Sentinel-1 berbasis radar dapat digunakan sebagai alternatif karena tidak terpengaruh awan seperti sensor optik.